<a href="https://colab.research.google.com/github/yangjl96github/gpt-ai-assistant/blob/main/%E6%A8%82%E5%B1%8B%E7%B6%B2%E7%88%AC%E8%9F%B2%E7%A8%8B%E5%BC%8F%E7%A2%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import time
from google.colab import files

# 設定 API 網址 (不要帶參數，參數我們用 params 字典來傳遞比較乾淨)
api_url = "https://realtyprice.rakuya.com.tw/realprice-search/api/realprice-list-search/index"

# 建立一個 list 來儲存所有頁面的資料
all_data = []

# 設定 headers，偽裝成正常瀏覽器
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Referer": "https://realtyprice.rakuya.com.tw/realprice/result?zipcode=202&sell_type=apartment&age=30~&exclude_special=1&sort=10"
}

# 初始設定
current_page = 1
max_pages = 5  # 【注意】這裡我先設定只抓 5 頁讓你測試。確定沒問題後，你可以把它改成 150 (大於總頁數 108 即可)，程式到底會自動停止。

print("開始抓取資料...")

while current_page <= max_pages:
    # 動態設定每次請求的參數
    params = {
        "zipcode": "202",
        "sell_type": "apartment",
        "age": "30~",
        "exclude_special": "1",
        "sort": "10",
        "page": current_page,
        "page_size": 20
    }

    response = requests.get(api_url, params=params, headers=headers)

    if response.status_code == 200:
        json_data = response.json()

        # 確保回傳狀態是成功的，且有 data 欄位
        if json_data.get("status") and "data" in json_data and "items" in json_data["data"]:
            items = json_data["data"]["items"]

            # 如果抓不到 items，代表已經翻到最後一頁了，跳出迴圈
            if len(items) == 0:
                print("已經沒有更多資料，抓取完畢。")
                break

            all_data.extend(items)
            print(f"成功抓取第 {current_page} 頁，目前共累積 {len(all_data)} 筆資料。")
            current_page += 1
        else:
            print(f"第 {current_page} 頁資料格式異常，停止抓取。")
            break
    else:
        print(f"請求失敗，狀態碼：{response.status_code}")
        break

    # 【強烈建議】每次請求後暫停 1.5 秒，避免對網站伺服器造成負擔而被鎖 IP
    time.sleep(1.5)

# 抓取結束後，將資料轉成 Excel 並下載
if all_data:
    print("\n資料整理中，準備輸出 Excel...")

    # 1. 將所有 JSON 物件轉為 pandas DataFrame
    df = pd.DataFrame(all_data)

    # 2. 儲存為 Excel 檔案
    filename = "rakuya_real_price.xlsx"
    df.to_excel(filename, index=False)

    print(f"儲存成功！檔案名稱：{filename}，準備觸發下載...")

    # 3. 呼叫 Colab 內建函式，將檔案下載到你的本機端
    files.download(filename)
else:
    print("沒有抓取到任何資料，請檢查網址或參數設定。")